# Time series
Have a look at the code below using generated temperature as sensor data. A Regressor should be used to forecast temperature.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt

Creation of dataset:

In [ ]:
# create synthetic time series of temperature
np.random.seed(42)
n = 500

time = np.arange(n)
amplitude = 20*np.random.choice(np.random.random_sample(size=5), size=n)
signal = np.sin(time * 0.03)*amplitude+np.random.randint(0, 10) # simulate temperature
noise = np.random.normal(0, 0.2, n)

y = signal + noise

df = pd.DataFrame({"y": y})
df.describe()

Create features (lag features), feature matrix X and target vector y. 

In [ ]:
# create lag features
df["lag1"] = df["y"].shift(1)
df["lag2"] = df["y"].shift(2)
df["lag3"] = df["y"].shift(3)

df = df.dropna()

X = df[["lag1", "lag2", "lag3"]]
y = df["y"]



Train-test split! Note that instead of a random split, a simple temporal slicing is done!

In [ ]:
# time based split
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]


Train a RF Regressor

In [ ]:
# train model
model = RandomForestRegressor()
model.fit(X_train, y_train)

# predictions
preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)

In [ ]:
sns.lineplot(df["y"])

Try to improve the accuracy of the Regressor by adding lag features or a rolling mean and rolling standard, play with the window and distances of lag features:

In [ ]:
# create lag features
df["lag1"] = df["y"].shift(1)
df["lag2"] = df["y"].shift(2)
df["lag3"] = df["y"].shift(3)


df["rolling_mean_5"] = df["y"].rolling(window=5).mean()
df["rolling_std_5"] = df["y"].rolling(window=5).std()

df = df.dropna()

X = df[["lag1", "lag2", "lag3", "rolling_std_5", "rolling_mean_5"]]
y = df["y"]

# time based split
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

You can also try other Models (e.g. GradientBoostingRegressor or LinearRegression) and see how the RMSE behaves...

### Forecasting:

Assume you want to forecast the next three temperature points. There are two different ways to achieve that - recursive or direct. Have a look at the code below and discuss what it achieves:

In [104]:
df = pd.DataFrame({"y": y})

df["lag1"] = df["y"].shift(1)
df["lag2"] = df["y"].shift(2)
df["lag3"] = df["y"].shift(3)

df["y_t1"] = df["y"].shift(-1)
df["y_t2"] = df["y"].shift(-2)
df["y_t3"] = df["y"].shift(-3)

df = df.dropna()

X = df[["lag1","lag2","lag3"]]

y1 = df["y_t1"]
y2 = df["y_t2"]
y3 = df["y_t3"]

split = int(len(df)*0.8)

X_train = X[:split]
X_test = X[split:]

model1 = RandomForestRegressor().fit(X_train, y1[:split])
model2 = RandomForestRegressor().fit(X_train, y2[:split])
model3 = RandomForestRegressor().fit(X_train, y3[:split])

pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)
pred3 = model3.predict(X_test)